In [31]:
# Preparing pathing
%load_ext autoreload
%autoreload 2
from titanic_ml import paths
import matplotlib.pyplot as plt
import pandas as pd
from titanic_ml.common.data.eda import summarize_categorical_column, summarize_numerical_column
from titanic_ml.common.data.eda import run_eda 
TARGET = "Survived"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Experiment =[
    Name,
    Model_name,
    Model_params:{},
    feature_engineering:{},
    features:{numerical:{}, onehot:{}, ordinal:{}},
    preprocessing:{},
    evaluation:{},
    notes,
]

### Run_Experiments Blueprint - WIP:

run_experiments (Df, Experiments) -> Result_df for each experiment:

    Feature engineering function (DF, Experiments[feature_engineering]) -> this experiments modified_df:
        (creates/modifies columns)

    build_preprocessor(Experiment[[features, preprocessing,]]) -> preprocessor:
        (prepares selected columns for sklearn model)

    model(modified_df, preprocessor, Experiment[Model_name, Model_params, evaluation]) -> Prediction and evaluation:
        (trains/predicts/evaluates)

In [32]:
from titanic_ml.common.experiments.runner import run_experiments
from titanic_ml.common.experiments import experiment_config
from titanic_ml.common.experiments.experiment_report import experiment_report

df = pd.read_csv(paths.TRAIN_PATH)
# print(df.head())

# working_df = Titanic_feature_engineering(df)
# print(working_df.head())

config = experiment_config.baseline_config
# print(baseline)




In [46]:
def add_family_features(df):
    df = df.copy()

    df['Family_size'] = df['SibSp'] + df['Parch'] + 1
    df['Alone'] = (df['Family_size'] == 1).astype(int)
    
    return df

In [47]:
def add_title(df):
    df = df.copy()

    title_name = df['Name'].str.split(',').str[1]
    title = title_name.str.split('.').str[0]
    df['Title'] = title.str.strip()
    df['Title'] = df['Title'].replace({'Mlle':'Miss', 'Ms':'Miss', 'Mme': 'Mrs'})
    
    return df

def group_rare_title(df):
    df = df.copy()

    rare_titles = [
    'Dr', 'Rev', 'Col', 'Major', 'Don', 'Lady',
    'Sir', 'Capt', 'the Countess', 'Jonkheer'
    ] 
    df['Title'] = df['Title'].replace(rare_titles, 'Rare')
    
    return df

def add_full_title_feature(df):
    df = add_title(df)
    df = group_rare_title(df)

    return df

In [61]:
def add_has_cabin(df):
    df = df.copy()

    df['Has_Cabin'] = df['Deck'].notnull().astype(int)
    # Filling Deck's Nan with 'None'
    df['Deck'] = df['Deck'].fillna('None')

    return df

def add_deck(df):
    df = df.copy()
    df['Deck'] = df['Cabin'].str[0]
    return df

def add_full_deck(df):
    df = add_deck(df)
    df = add_has_cabin(df)

    return df

In [66]:
df = pd.read_csv(paths.TRAIN_PATH)
exp = {'fn':[add_family_features, add_full_deck, add_full_title_feature]}
working_df = df.copy()
for fn in exp.get('fn',[]):
    working_df = fn(working_df)
working_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_size,Alone,Deck,Has_Cabin,Title
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,2,0,None,0,Mr
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,2,0,C,1,Mrs
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1,1,None,0,Miss
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,2,0,C,1,Mrs
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,1,1,None,0,Mr


In [42]:
from titanic_ml.common.data.eda import summarize_categorical_column

title = summarize_categorical_column(test_df, 'Cabin')

In [43]:
print(title)

            count percent
MISSING       687    77.1
G6              4    0.45
C23 C25 C27     4    0.45
B96 B98         4    0.45
F2              3    0.34
D               3    0.34
E101            3    0.34
C22 C26         3    0.34
F33             3    0.34
C83             2    0.22
...           ...     ...
B102            1    0.11
B69             1    0.11
E49             1    0.11
C47             1    0.11
D28             1    0.11
E17             1    0.11
A24             1    0.11
C50             1    0.11
B42             1    0.11
C148            1    0.11


In [40]:
result = run_experiments(df, config, target=TARGET, verbose=True)

Running experiment: baseline_logreg

Experiment 'baseline_logreg' results:
  model_name: logreg
  status: success
  error_type: 
  error_message: 
  test_accuracy_mean: 0.786
  test_accuracy_std: 0.018
  train_accuracy_mean: 0.803
  train_accuracy_std: 0.005
  test_precision_mean: 0.736
  test_precision_std: 0.036
  train_precision_mean: 0.762
  train_precision_std: 0.011
  test_recall_mean: 0.693
  test_recall_std: 0.038
  train_recall_mean: 0.708
  train_recall_std: 0.015
  test_f1_mean: 0.713
  test_f1_std: 0.026
  train_f1_mean: 0.734
  train_f1_std: 0.008
  fit_time_mean: 0.024
  score_time_mean: 0.017
  notes: Base logistic regression. Baseline for comparison.

Experiment 'baseline_logreg' completed.
----------------------------------------
Running experiment: baseline_knn

Experiment 'baseline_knn' results:
  model_name: knn
  status: success
  error_type: 
  error_message: 
  test_accuracy_mean: 0.809
  test_accuracy_std: 0.021
  train_accuracy_mean: 0.861
  train_accuracy_std:

In [41]:
individual_report, full_report = experiment_report(result, config, print_report=True)

Individual Experiment Reports:

Experiment - baseline_logreg:
| Field | Value |
|---|---|
|Train accuracy| 0.803 ± 0.005 |
|Train precision| 0.762 ± 0.011 |
|Train recall| 0.708 ± 0.015 |
|Train f1| 0.734 ± 0.008 |
|Test accuracy| 0.786 ± 0.018 |
|Test precision| 0.736 ± 0.036 |
|Test recall| 0.693 ± 0.038 |
|Test f1| 0.713 ± 0.026 |

Full configuration:
```python
{'name': 'baseline_logreg',
 'features': ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'],
 'feature_engineering': None,
 'preprocessing': {'numeric_features': ['Age', 'SibSp', 'Parch', 'Fare'],
                   'onehot_features': ['Sex', 'Embarked'],
                   'ordinal_features': ['Pclass'],
                   'numeric_imputer': 'median',
                   'categorical_imputer': 'most_frequent',
                   'scaler': 'standard'},
 'model_name': 'logreg',
 'model_params': {'max_iter': 1000, 'random_state': 42},
 'evaluation': {'method': 'cross_validation',
                'cv': 5,
            